# Introduction

The purpose of this notebook is to design a basic Neural Network (multi-layer perceptron) from scratch.

# Import

In [1]:
import numpy as np
from pydantic import BaseModel, Field, computed_field
from typing import Union, Callable, Literal, List

In [ ]:
class BaseActivation(BaseModel):
    """Base class for activation functions. Must implement forward and backward methods.
    """

    def forward(self, x: np.ndarray) -> np.ndarray:
        raise NotImplementedError("forward method must be implemented in the subclass")

    def backward(self, x: np.ndarray) -> np.ndarray:
        raise NotImplementedError("backward method must be implemented in the subclass")

class SigmoidActivation(BaseActivation):
    """Sigmoid activation function.
    """

    def forward(self, x: np.ndarray) -> np.ndarray:
        return 1 / (1 + np.exp(-x))

    def backward(self, x: np.ndarray) -> np.ndarray:
        sig = self.forward(x)
        return sig * (1 - sig)

class ReLUActivation(BaseActivation):
    """ReLU activation function.
    """

    def forward(self, x: np.ndarray) -> np.ndarray:
        return np.maximum(0, x)

    def backward(self, x: np.ndarray) -> np.ndarray:
        return (x > 0).astype(float)

In [30]:
class BaseLoss(BaseModel):
    """Base class for loss functions. Must implement forward and backward methods.
    """

    def forward(self, y_true: np.ndarray, y_pred: np.ndarray) -> float:
        raise NotImplementedError("forward method must be implemented in the subclass")

    def backward(self, y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
        raise NotImplementedError("backward method must be implemented in the subclass")

class MeanSquaredErrorLoss(BaseLoss):
    """Mean Squared Error loss function.
    """

    def forward(self, y_true: np.ndarray, y_pred: np.ndarray) -> float:
        return np.mean((y_true - y_pred) ** 2)

    def backward(self, y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
        return 2 * np.sum(y_pred - y_true, axis=0) / y_true.shape[0]

In [ ]:
class Layer(BaseModel):
    type: Literal['input', 'hidden', 'output']
    nodes: int = Field(gt=0, default=5)
    activation: Union[Literal['sigmoid', 'relu'], type[BaseActivation]] = None
    # TODO: Add 'softmax' option
    # TODO: activation should not be none if type is 'hidden' or 'output'
    W: List = None
    b: List = None
    Z: List = None
    A: List = None

    @computed_field
    @property
    def activation_obj(self) -> BaseActivation:
        if self.activation == 'sigmoid':
            return SigmoidActivation()
        elif self.activation == 'relu':
            return ReLUActivation()
        elif isinstance(self.activation, BaseActivation):
            return self.activation
        else:
            raise ValueError(f"Unsupported activation type: {self.activation}. Supported types are 'sigmoid', 'relu', or an INSTANTIATED OBJECT of a subclass of BaseActivation.")

class MultiLayerPerceptron(BaseModel):
    layers: list[Layer] = Field(min_length=2)
    loss: Union[Literal['mse'], type[BaseLoss]] = 'mse'
    train_losses: list[float] = []

    @computed_field
    @property
    def loss_obj(self) -> type[BaseLoss]:
        if self.loss == 'mse':
            return MeanSquaredErrorLoss
        elif isinstance(self.loss, type) and issubclass(self.loss, BaseLoss):
            return self.loss
        else:
            raise ValueError(f"Unsupported loss type: {self.loss}. Supported types are 'mse' or a subclass of BaseLoss.")

    def __init__(self, **kwargs):
        print("Initializing MultiLayerPerceptron...")
        super().__init__(**kwargs)
        self.validate_layers(self.layers)
        self.initialize_weights()    

    def validate_layers(self, layers: list[Layer]):
        print("Validating layers...")
        if layers[0].type != 'input':
            raise ValueError("The first layer must be of type 'input'.")
        if layers[-1].type != 'output':
            raise ValueError("The last layer must be of type 'output'.")
        for i in range(1, len(layers)-1):
            if layers[i].type != 'hidden':
                raise ValueError("All intermediate layers must be of type 'hidden'.")

    def initialize_weights(self):
        for i in range(1, len(self.layers)):
            print(f"Initializing weights for layer {i} with shape ({self.layers[i-1].nodes}, {self.layers[i].nodes})...")
            self.layers[i].W = np.random.randn(self.layers[i-1].nodes, self.layers[i].nodes)
            self.layers[i].b = np.zeros((1, self.layers[i].nodes))

    def forward(self, x: np.ndarray) -> np.ndarray:
        """Perform a forward pass through the network."""
        for i, layer in enumerate(self.layers):
            if layer.type == 'input':
                if x.shape[1] != layer.nodes:
                    raise ValueError(f"Input data must have {layer.nodes} features, but got {x.shape[1]}.")
                else:
                    layer.A = x
                    continue
            print(f"input shape: {self.layers[i-1].A.shape}")
            print(f"W shape: {layer.W.shape}")
            layer.Z = self.layers[i-1].A @ layer.W + layer.b
            layer.A = layer.activation_obj.forward(layer.Z)
        return self.layers[-1].A

    def backward(self, alpha: float, y_true: np.ndarray):
        """Perform a backward pass through the network."""
        # dL/dW -> dW = dZ/dW -> input of prev layer * dL/dZ -> dZ
        # dL/dZ -> dZ = dA/dZ -> differentiate activation function wrt Z * dL/dA -> differentiate loss function wrt A (dA)
        dA = self.loss_obj().backward(y_true, self.layers[-1].A)
        for i in reversed(range(1, len(self.layers))):
            layer = self.layers[i]
            prev_layer = self.layers[i-1]
            dZ = layer.activation_obj.backward(layer.Z) * dA
            dW = prev_layer.A.T @ dZ
            db = np.sum(dZ, axis=0, keepdims=True)
            dA = dZ @ layer.W.T
            # Update weights and biases
            layer.W -= alpha * dW
            layer.b -= alpha * db

    def fit(self, X: np.ndarray, y: np.ndarray, epochs: int = 1000, alpha: float = 0.01):
        for epoch in range(epochs):
            y_hat = self.forward(X)
            self.train_losses.append(self.loss_obj().forward(y, y_hat))
            self.backward(alpha, y)

    def predict(self, X: np.ndarray) -> np.ndarray:
        return self.forward(X)

In [39]:
layers = [
    Layer(type='input', nodes=3, activation='relu'),
    Layer(type='hidden', nodes=4, activation='sigmoid'),
    Layer(type='hidden', nodes=5, activation='sigmoid'),
    Layer(type='output', nodes=1, activation='sigmoid'),
]
mlp = MultiLayerPerceptron(layers=layers)


Initializing MultiLayerPerceptron...
Validating layers...
Initializing weights for layer 1 with shape (3, 4)...
Initializing weights for layer 2 with shape (4, 5)...
Initializing weights for layer 3 with shape (5, 1)...


In [48]:
y_hat = mlp.forward(np.array([[0.1, 0.2, 0.3], [0.5, 0.4, 0.3]]))
y_hat
for layer in mlp.layers:
    print(layer.W)

input shape: (2, 3)
W shape: (3, 4)
input shape: (2, 4)
W shape: (4, 5)
input shape: (2, 5)
W shape: (5, 1)
None
[[ 0.55122997  0.59085713  1.12834798  0.55343277]
 [-0.91401344  2.32365206 -0.04130513 -1.22027931]
 [-1.31888495 -0.6071502  -1.51280527 -0.1162494 ]]
[[-0.63423574 -1.04276793  0.16018202  0.06778593 -1.0974138 ]
 [-0.73931126 -1.42747913  1.77083647  1.40431     1.49032077]
 [ 2.00358738 -0.51561956 -0.32554417  0.30998347 -0.14215276]
 [-1.32075542  0.45472415  0.52073294  0.70492036  0.54082282]]
[[-1.23861116]
 [-0.19534371]
 [-1.33285709]
 [-0.75699606]
 [-0.77623323]]


In [49]:
mlp.backward(alpha=0.01, y_true=np.array([[1], [0]]))

In [50]:
for layer in mlp.layers:
    print(layer.W)

None
[[ 0.5512548   0.59082629  1.12830961  0.55344017]
 [-0.91398801  2.32361804 -0.04134386 -1.22027233]
 [-1.31885892 -0.6071874  -1.51284435 -0.11624286]]
[[-0.63434979 -1.04277997  0.16009655  0.06774019 -1.0974785 ]
 [-0.73951066 -1.42749997  1.77068827  1.40423075  1.4902082 ]
 [ 2.00344628 -0.51563431 -0.32564905  0.30992739 -0.14223241]
 [-1.32088928  0.45470999  0.52063249  0.7048666   0.54074682]]
[[-1.23819833]
 [-0.19513704]
 [-1.33206015]
 [-0.75618236]
 [-0.77554705]]


Questions:
1. Can a polynomial equation of degree= 2, be represented by a neural network with a single hidden layer (or multiple hidden layers) by a linear threshold activation?
    a. if we are modeling a y=x^2 pattern, can a Neural Net pick up on the pattern?.